In [4]:
from pathlib import Path
import csv
import json
from collections import defaultdict

ROOT = Path("../")
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "dataset"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP = {"favor": 0, "against": 1, "none": 2}
SPLITS = {
    "train": "train.json",
    "valid": "valid.json",
    "test": "test.json",
}


def load_text_mapping(text_csv_path: Path):
    """Load post/comment text by conversation index prefix from text.csv."""
    mapping = {}
    with text_csv_path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            mapping[str(row["id"]).strip()] = str(row["text"]).strip()
    return mapping


def build_samples(target_dir: Path, split_file: Path, id_start: int):
    text_map = load_text_mapping(target_dir / "text.csv")
    with split_file.open("r", encoding="utf-8") as f:
        items = json.load(f)

    all_label_dict = {}
    for item in items:
        index = item["index"][-1]
        label = LABEL_MAP[item["stance"]]
        if index in all_label_dict:
            print(f"{index} already in all_label_dict")
        all_label_dict[index] = label

    samples = []
    for sample_id, item in enumerate(items):
        indices = [str(x) for x in item["index"]]
        sentences = [text_map[idx] for idx in indices]

        label = LABEL_MAP[item["stance"]]
        samples.append(
            {
                "id": id_start + sample_id,
                "target": target_dir.name,
                "target_type": "n",
                "sentences": sentences,
                "speakers": list(range(len(sentences))),
                "label": label,
                "all_labels": [all_label_dict[idx] for idx in indices],
                "last_speaker_sentences": [],
                "last_speaker_sentences_labels": [],
            }
        )
    return samples

all_samples = defaultdict(list)
for target_dir in sorted([p for p in DATA_DIR.iterdir() if p.is_dir()]):
    for split_name, split_filename in SPLITS.items():
        split_path = target_dir / split_filename
        if not split_path.exists():
            continue
        
        id_start = len(all_samples[split_name])
        samples = build_samples(target_dir, split_path, id_start)
        all_samples[split_name].extend(samples)

for split_name, samples in all_samples.items():
    out_path = OUT_DIR / f"{split_name}.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(samples)} samples to {out_path}")

Wrote 10925 samples to ..\dataset\train.json
Wrote 2209 samples to ..\dataset\valid.json
Wrote 2371 samples to ..\dataset\test.json
